# Theil-Sen & Mann-Kendall Trendanalys (Pixelvis)

Den här notebooken utgör **huvudanalyssteget** i kedjan för att förstå långsiktiga utvecklingstrender. Scriptet tar de preprocessade satellitindexen (genererade i `preprocessing.ipynb`) och skapar pixelbaserade regressionsberäkningar över all tillgänglig tid för att spåra riktning och signifikans.

### Arbetsflöde och struktur

Arbetsflödet sker i 7 huvudsakliga steg för att säkerställa statistisk robusthet och smidig filutrullning:

---

### 1. Konfiguration och scenurval
Studieområde (t.ex. `LF` eller `GRM`) konfigureras och onödig/förorenade gamla resultat rensas. CSV-loggarna från preprocessingen kombineras för att sedan systematiskt välja ut **en molnfri scen per år**. Algoritmen för selektion prioriterar scener baserat på högsommarperioden:

| Prioritet | Datumintervall |
|-----------|---------------|
| 1 (bäst) | 20 juni – 15 augusti |
| 2 | 16–31 augusti |
| 3 | 1–19 juni |

Inom varje prioritetsnivå väljs scenen närmast referensdatumet **15 juli**. Vi filtrerar även specifika år/satelliter (t.ex. Landsat 7-ränder eller extremåren 2025).

---

### 2. Bygg Datakuber (Xarray Stacks)
De "bästa" utvalda tids-bilderna läses in och sammanfogas till tredimensionella datakuber `(tid, y, x)` för repektive spektralt index (NDVI, NDWI, MNDWI m.m.). Detta säkerställer att vi kan spåra exakt samma pixel genom tidens gång.

---

### 3. Autokorrelationsanalys
Innan Theil-Sen körs, säkerställer koden att vi väljer det statistiskt mest korrekta **Mann-Kendall**-testet. Tidsseriedata lider ofta av autokorrelation (där årets värde vilar på fjolårets), vilket kan ge oss falska positiva trender (p-värden p.g.a. standardfel). 
Koden plottar en ACF-graf (Autocorrelation Function) och kör ett **Ljung-Box-test** varvid scriptet rekommenderar oss vilket Mann-Kendall-bibliotek vi bör införa i steg 4 (oftast `hamed_rao_modification_test`).

---

### 4. Pixelvis Trendekvation (Theil-Sen + Mann-Kendall)
Scriptet itereras spatialt ned på pixelnivå och tillämpar en dubbel testrutin:
- **Mann-Kendall test**: Bekräftar om trenden är statistiskt signifikant eller ej.
- **Theil-Sen estimator (Slope)**: Beräknar lutning/förändring per år (robust mot outliers).

Resultaten för varje pixel (`slope`, `pvalue`, `tau`, `z`) matas därefter in i nya GeoTIFF-raster.

---

### 5. Rasterklassificering (Signifikans vs Förändring)
Här slås `slope` och `pvalue`-rastren ihop för att visa en kategorisk skala. En pixel tilldelas ett värde mellan `-4` och `+4` (t.ex. `-4` betyder "Stark minskning, p<0.01", och `+4` betyder "Stark ökning, p<0.01". Rastret underlättar för visuell granskning i GIS-program.

---

### 6. Statistisk Sammanfattning
Rastrens värden slås samman i en tabell över hela studieområdet. Den sammanfattar hur många procent av markens pixlar som faktiskt blir blötare/torrare eller ökar/minskar i biomassa, och räknar ut områdets generella median-lutning.

---

### 7. Sammanslagning av flera Studieområden
I det sista steget körs sammanslagningskod för att konkatenera de olika `csv`-filerna vi presterat för varje studieområde in i en master-sammanfattning.

## Välja scener och bygg data arrays

In [ ]:

import os
import glob
import numpy as np
import xarray as xr
import pandas as pd
import seaborn as sns
import geopandas as gpd
import rioxarray as rxr
import pymannkendall as mk
import matplotlib.pyplot as plt

from statsmodels.graphics.tsaplots import plot_acf
from statsmodels.stats.diagnostic import acorr_ljungbox 

from config import (
    build_analysis_output_dirs,
    build_analysis_satellites,
    build_stack,
    load_and_prepare_scene_logs,
    print_satellite_setup,
    select_best_scenes,
    OPEN_WETLAND_AREA
    )


In [ ]:
# ============================================
# Definera samtliga studieområdes-variabler
# ============================================

area = "LF"  # OBS: här väljs studieområdet som en variabel, viktig för att bygga stackarna nedan!
sensor = "landsat"  # "landsat" eller "sentinel2"

# Hämtad från config.py, där satellitdata och sökvägar är definierade per studieområde
satellites = build_analysis_satellites(area, sensor=sensor)

# Output-mapp för theil_sen_mk-resultat, per studieområde
analysis_output_dirs = build_analysis_output_dirs(sensor=sensor)
output_dir_theil_sen_mk  = analysis_output_dirs["theil_sen_vizualization"]

print_satellite_setup(area, satellites, sensor=sensor)

In [ ]:

# =============================================
# Töm theil_sen_mk-mappen för aktuellt studieområde (om clean_output är True)
# OBS: Raderar slope- och pvalue-filer för aktuellt area
# =============================================
clean_output = True

if clean_output:
    confirm = input(f"Är du säker på att du vill tömma TS/MK mappen för {area}? (y/n): ")
    if confirm.lower() == "y":
        skipped = []
        for tif_path in glob.glob(os.path.join(output_dir_theil_sen_mk, f"{area}_*.tif")):
            try:
                os.remove(tif_path)
            except PermissionError:
                skipped.append(os.path.basename(tif_path))
                print(f"  ⚠ Kunde inte radera (fil används): {os.path.basename(tif_path)}")
        if skipped:
            print(f"OBS: {len(skipped)} fil(er) kunde inte raderas – stäng QGIS/Utforskaren eller starta om kerneln.")
        else:
            print(f"theil_sen_mk tömda för {area}.")
    else:
        print("Avbröt – inga filer raderades.")


In [ ]:

# =============================================
# Steg 1: Läs in och kombinera loggar (utskrift från preprocessing)
# =============================================
log = load_and_prepare_scene_logs(satellites, area)

# Valbart inför körning: sätt True för att exkludera enbart Landsat 7
exclude_l07_this_run = True

if exclude_l07_this_run:
    before_n = len(log)
    if "sat_name" in log.columns:
        # Strikt filter: ta bara bort L07-rader, behåll L05/L0809
        log = log[log["sat_name"].astype(str).str.upper() != "L07"].copy()
    elif "scene_name" in log.columns:
        # Fallback om sat_name saknas i loggen
        log = log[~log["scene_name"].astype(str).str.contains("LE07", case=False, na=False)].copy()
    print(f"L07 exkluderat denna körning: {before_n - len(log)} scener borttagna")


In [ ]:
# =============================================
# Steg 2: Välj bästa molnfria scen per år
# =============================================
best_scenes = select_best_scenes(log)
exclude_years = [2025] # Eller 2022-2025 för GM
best_scenes = best_scenes[~best_scenes["date"].dt.year.isin(exclude_years)]  # Exkludera 2022-2025 eller 2025 från trenden

# Skapar en DataFrame med de valda scenerna
print(f"Antal valda scener: {len(best_scenes)}")
print(best_scenes[["year", "date", "days_from_july15", "sat_name", "cloud_pct"]])

# Spara som CSV
out_path = os.path.join(output_dir_theil_sen_mk, f"{area}_best_scenes.csv")
best_scenes.to_csv(out_path, index=False)
print(f"Sparad: {out_path}")


In [ ]:
# =============================================
# Steg 3: Bygg xarray stack per index
# =============================================

# Förbered Shapefilen om området är TAM
clip_shp = r"D:\Masteruppsats\Studieomraden\Omr_polygon\TAM_lake.shp"
clip_gdf = None
if area == "TAM" and os.path.exists(clip_shp):
    clip_gdf = gpd.read_file(clip_shp)

# Bygg index-stack
stacks = {}
for index_name in ["NDVI", "MNDWI", "NDMI", "NDWI", "EVI"]: # Kan lägga till fler index senare --- IGNORE --- 
    print(f"Bygger stack för {index_name}...")
    da = build_stack(best_scenes, index_name, area) # OBS: här väljs "area" som en variabel, se nedan
    
    # Klipp stacken om area är TAM - Behåll det som är utanför shapefilen
    if clip_gdf is not None:
        # Säkerställ att CRS (koordinatsystem) matchar
        clip_gdf_match = clip_gdf.to_crs(da.rio.crs)
        # invert=True betyder att pixlar *innanför* shapefilen sätts till NoData, resten blir kvar
        da = da.rio.clip(clip_gdf_match.geometry, clip_gdf_match.crs, invert=True)
        print("  Maskat bort (tagit bort) området innanför TAM_lake.shp")

    stacks[index_name] = da
    print(f"  Shape: {stacks[index_name].shape}")

In [ ]:
# ============================================
# Kontrollera data array och max/min-värden i stacken
# ============================================
print(stacks["EVI"].head())
print(f'Maxvärdet är {stacks["EVI"].max().values} och borde vara <= 1.0')
print(f'Minvärdet är {stacks["EVI"].min().values} och borde vara >= -1.0')

## Autokorrelationsanalys – vilket MK-test ska användas?


Autokorrelation i en tidsserie kan göra att det **standard-MK-testet** (`original_test`) ger felaktiga p-värden (för låga → för många falska positiver).  
Rekommendation baserat på lag-1 autokorrelation:

| Situation | Rekommenderat test |
|---|---|
| Ingen signifikant autokorrelation | `original_test` |
| Positiv autokorrelation | `hamed_rao_modification_test` eller `yue_wang_modification_test` |
| Stark trend + autokorrelation | `trend_free_pre_whitening` |
| Säsongsdata | `seasonal_test` |

Cellen nedan beräknar ACF (autokorrelationsfunktion) för varje index (spatialt medelvärde) och testar om autokorrelationen är signifikant med **Ljung-Box-testet**.


In [ ]:

# =============================================
# Autokorrelationsanalys per index. Kontrollerar autokorrelation inom hela polygonen.
# =============================================
# OBS: Använder spatialt medelvärde per tidssteg som proxy-tidsserie.
# Lag-1 autokorrelation och Ljung-Box-test (lag 1–3) avgör val av MK-test.
# =============================================

ALPHA = 0.05  # Signifikansnivå
N_LAGS = min(10, len(next(iter(stacks.values())).time) // 2)  # Max lags för ACF-plot

fig, axes = plt.subplots(len(stacks), 1, figsize=(10, 3.5 * len(stacks)))
if len(stacks) == 1:
    axes = [axes]

recommendations = {}

for ax, (index_name, stack) in zip(axes, stacks.items()):
    # Beräkna spatialt medelvärde per år
    ts = stack.mean(dim=["x", "y"], skipna=True).to_series().dropna()
    n = len(ts)

    # ACF-plot
    plot_acf(ts, lags=N_LAGS, alpha=ALPHA, ax=ax, title=f"ACF – {index_name}  (n={n})")
    ax.axhline(0, color="black", linewidth=0.8)
    ax.set_xlabel("Lag (yr)")
    ax.set_ylabel("Autocorrelation")
    ax.grid(True, linestyle="--", alpha=0.4)

    # Ljung-Box-test på lag 1–3
    max_lb_lag = min(3, n - 2)
    lb_result = acorr_ljungbox(ts, lags=max_lb_lag, return_df=True)
    min_pval = lb_result["lb_pvalue"].min()

    # Lag-1 autokorrelation
    lag1_acf = ts.autocorr(lag=1)

    # Rekommendation
    if min_pval < ALPHA:
        if lag1_acf > 0:
            rec = "hamed_rao_modification_test  (positiv autokorrelation)"
        else:
            rec = "original_test (negativ autokorr. – MK är konservativt)"
    else:
        rec = "original_test  (ingen signifikant autokorrelation)"

    recommendations[index_name] = {
        "n": n,
        "lag1_acf": round(lag1_acf, 4),
        "ljungbox_min_p": round(min_pval, 4),
        "signifikant_autokorr": min_pval < ALPHA,
        "rekommenderat_test": rec,
    }

plt.tight_layout()

# Spara ACF-plot som bild (måste vara före plt.show())
out_path_fig = os.path.join(output_dir_theil_sen_mk, f"{area}_acf_plot.png")
fig.savefig(out_path_fig, dpi=300, bbox_inches="tight")
print(f"Sparad: {out_path_fig}")

plt.show()

# Sammanfattning
print("=" * 70)
print(f"{'Index':<10} {'n':>4}  {'Lag-1 ACF':>10}  {'LjungBox p':>11}  {'Signif.':>8}")
print("-" * 70)
for idx, vals in recommendations.items():
    sig_str = "JA  ⚠" if vals["signifikant_autokorr"] else "nej"
    print(f"{idx:<10} {vals['n']:>4}  {vals['lag1_acf']:>10.4f}  {vals['ljungbox_min_p']:>11.4f}  {sig_str:>8}")
print("=" * 70)
print()
for idx, vals in recommendations.items():
    print(f"  {idx}: → {vals['rekommenderat_test']}")

# Spara ACF-resultat som CSV
df_acf = pd.DataFrame(recommendations).T.reset_index().rename(columns={"index": "index_name"})
out_path_csv = os.path.join(output_dir_theil_sen_mk, f"{area}_acf_results.csv")
df_acf.to_csv(out_path_csv, index=False)
print(f"Sparad: {out_path_csv}")

## Theil-Sen och Mann Kendall

In [ ]:
def theil_sen_pixelwise(stack, mk_test_func=mk.original_test):
    """
    Kör Mann-Kendall + Theil-Sen på varje pixel.

    Parametrar
    ----------
    stack        : xarray DataArray med dimensioner (tid, y, x)
    mk_test_func : MK-testfunktion från pymannkendall (default: mk.original_test)
                   Alternativ: mk.hamed_rao_modification_test, mk.yue_wang_modification_test

    Returnerar
    ----------
    slope     : Theil-Sen lutning per pixel (förändring per år)
    intercept : Theil-Sen intercept per pixel
    pvalue    : p-värde från MK-testet per pixel
    tau       : Kendalls Tau per pixel
    z         : Z-statistika per pixel
    """
    values    = stack.values  # (tid, y, x)
    slope     = np.full(values.shape[1:], np.nan)
    intercept = np.full(values.shape[1:], np.nan)
    pvalue    = np.full(values.shape[1:], np.nan)
    tau       = np.full(values.shape[1:], np.nan)
    z         = np.full(values.shape[1:], np.nan)

    # OBS: Dubbel for-loop fungerar bra för små områden (~1500 pixlar).
    # För större områden kan xr.apply_ufunc användas för att vektorisera och snabba upp.
    for i in range(values.shape[1]):
        for j in range(values.shape[2]):
            ts = values[:, i, j]
            # Hoppa över pixlar med för många NaN
            valid = ts[~np.isnan(ts)]
            if len(valid) < 10:  # Minst 10 observationer
                continue
            result = mk_test_func(valid)
            slope[i, j]     = result.slope
            intercept[i, j] = result.intercept
            pvalue[i, j]    = result.p
            tau[i, j]       = result.Tau
            z[i, j]         = result.z

    return slope, intercept, pvalue, tau, z

## Välj MK-test per index baserat på autokorrelationsanalysen.
Uppdatera mk_test-variabeln efter att ha kört autokorrelationsanalysen, se till att använda **samma trendtest till samtliga index.**

Tillgängliga test (från pymannkendall):
  * **mk.original_test**: standard, ingen autokorrelationskorrigering
  * **mk.hamed_rao_modification_test**: rekommenderas vid positiv autokorrelation
  * **mk.yue_wang_modification_test**: alternativ vid positiv autokorrelation
  * **mk.trend_free_pre_whitening_modification_test**: vid stark trend + autokorrelation
  ###### Source: https://github.com/mmhs013/pyMannKendall/blob/master/README.md

In [ ]:
# =============================================
# Steg 4: Kör Theil-Sen + MK och spara resultat
# =============================================

# Välj MK-test baserat på autokorrelationsanalysen ovan (används för samtliga index).
mk_test = mk.hamed_rao_modification_test

os.makedirs(output_dir_theil_sen_mk, exist_ok=True)

for index_name, stack in stacks.items():
    print(f"Kör Theil-Sen för {index_name} med {mk_test.__name__}...")
    slope, intercept, pvalue, tau, z = theil_sen_pixelwise(stack, mk_test_func=mk_test)

    # Använd spatial referens från stacken
    ref = stack.isel(time=0)

    for result_name, result_data in [
        ("slope",     slope),
        ("intercept", intercept),
        ("pvalue",    pvalue),
        ("tau",       tau),
        ("z",         z),
    ]:
        out_arr = xr.DataArray(
            result_data,
            dims=["y", "x"],
            coords={"y": ref.y, "x": ref.x}
        )
        out_arr = out_arr.rio.write_crs(stack.rio.crs)
        out_arr = out_arr.rio.write_nodata(np.nan)

        out_path = os.path.join(output_dir_theil_sen_mk, f"{area}_{index_name}_{result_name}.tif")
        if os.path.exists(out_path):
            os.remove(out_path)
        out_arr.rio.to_raster(out_path)
        print(f"  Sparad: {out_path}")

for index_name, stack in stacks.items():
    print(f"Åren som ingår i analysen är: {index_name}: {stack.time.min().dt.year.values}–{stack.time.max().dt.year.values}")

print("Klart!")

## Steg 8: Sammanfattning av MK/Theil-Sen-resultat

Läser in de sparade slope-, pvalue- och tau-rasters och beräknar pixelvis statistik per index:
- **Median slope** – typisk förändring per år
- **Median Tau** – styrka och riktning på trenden
- **Andel signifikanta pixlar** (p < 0,05)
- **Andel ökande/minskande** bland signifikanta pixlar
- **Dominant trend**

Resultatet sparas som CSV per studieområde. Kör för alla studieområden och konkatenera till en samlad fil.

In [ ]:
# =============================================
# Steg 8: Skapa slope+signifikans-raster
# =============================================
sig_level = 0.05
cmaps = {"EVI": "RdYlGn", "MNDWI": "RdBu", "NDMI": "BrBG", 
         "NDWI": "RdYlBu", "EVI": "RdYlGn"}

for index_name in stacks.keys():
    # Läs in slope och pvalue
    slope_arr  = rxr.open_rasterio(os.path.join(output_dir_theil_sen_mk, f"{area}_{index_name}_slope.tif")).squeeze()
    pvalue_arr = rxr.open_rasterio(os.path.join(output_dir_theil_sen_mk, f"{area}_{index_name}_pvalue.tif")).squeeze()

    slope  = slope_arr.values
    pvalue = pvalue_arr.values

    # =============================================
    # Raster 1: Slope maskerad med signifikans (p<0.05)
    # =============================================
    slope_masked = np.where(pvalue < sig_level, slope, np.nan)

    slope_masked_arr = xr.DataArray(
        slope_masked,
        dims=["y", "x"],
        coords={"y": slope_arr.y, "x": slope_arr.x}
    )
    slope_masked_arr = slope_masked_arr.rio.write_crs(slope_arr.rio.crs)
    slope_masked_arr = slope_masked_arr.rio.write_nodata(np.nan)

    out_path = os.path.join(output_dir_theil_sen_mk, 
                            f"{area}_{index_name}_slope_masked_p{int(sig_level*100)}.tif")
    if os.path.exists(out_path):
        os.remove(out_path)
    slope_masked_arr.rio.to_raster(out_path)
    print(f"Sparad: {out_path}")

    # =============================================
    # Raster 2: Kombinerad trend + signifikansklass (-4 till +4)
    # =============================================
    trend_class = np.full(slope.shape, np.nan)

    # Ökande trend
    trend_class = np.where((slope > 0) & (pvalue >= 0.10),  1, trend_class)
    trend_class = np.where((slope > 0) & (pvalue <  0.10),  2, trend_class)
    trend_class = np.where((slope > 0) & (pvalue <  0.05),  3, trend_class)
    trend_class = np.where((slope > 0) & (pvalue <  0.01),  4, trend_class)

    # Minskande trend
    trend_class = np.where((slope < 0) & (pvalue >= 0.10), -1, trend_class)
    trend_class = np.where((slope < 0) & (pvalue <  0.10), -2, trend_class)
    trend_class = np.where((slope < 0) & (pvalue <  0.05), -3, trend_class)
    trend_class = np.where((slope < 0) & (pvalue <  0.01), -4, trend_class)

    trend_arr = xr.DataArray(
        trend_class,
        dims=["y", "x"],
        coords={"y": slope_arr.y, "x": slope_arr.x}
    )
    trend_arr = trend_arr.rio.write_crs(slope_arr.rio.crs)
    trend_arr = trend_arr.rio.write_nodata(np.nan)

    out_path = os.path.join(output_dir_theil_sen_mk,
                            f"{area}_{index_name}_trend_significance.tif")
    if os.path.exists(out_path):
        os.remove(out_path)
    trend_arr.rio.to_raster(out_path)
    print(f"Sparad: {out_path}")

print("Klart!")

In [ ]:
# =============================================
# Steg 9: Sammanfattning av MK/Theil-Sen-resultat per index
# =============================================
summary_results = []

for index_name in stacks.keys():
    slope_arr  = rxr.open_rasterio(os.path.join(output_dir_theil_sen_mk, f"{area}_{index_name}_slope.tif")).squeeze()
    pvalue_arr = rxr.open_rasterio(os.path.join(output_dir_theil_sen_mk, f"{area}_{index_name}_pvalue.tif")).squeeze()
    tau_arr    = rxr.open_rasterio(os.path.join(output_dir_theil_sen_mk, f"{area}_{index_name}_tau.tif")).squeeze()

    slope_vals  = slope_arr.values.flatten()
    pvalue_vals = pvalue_arr.values.flatten()
    tau_vals    = tau_arr.values.flatten()

    mask = ~np.isnan(slope_vals)
    slope_vals  = slope_vals[mask]
    pvalue_vals = pvalue_vals[mask]
    tau_vals    = tau_vals[mask]

    sig_mask        = pvalue_vals < 0.05
    pct_significant = np.sum(sig_mask) / len(sig_mask) * 100

    sig_slopes = slope_vals[sig_mask]
    if len(sig_slopes) > 0:
        pct_increasing = np.sum(sig_slopes > 0) / len(sig_slopes) * 100
        pct_decreasing = np.sum(sig_slopes < 0) / len(sig_slopes) * 100
        if pct_increasing > pct_decreasing:
            dominant_trend = "Increasing"
        elif pct_decreasing > pct_increasing:
            dominant_trend = "Decreasing"
        else:
            dominant_trend = "No trend"
    else:
        dominant_trend = "No trend"
        pct_increasing = 0.0
        pct_decreasing = 0.0

    # Läs in trend_class-rastret för procent pixlar i varje klass
    trend_arr  = rxr.open_rasterio(os.path.join(output_dir_theil_sen_mk,
                 f"{area}_{index_name}_trend_significance.tif")).squeeze()
    trend_vals = trend_arr.values.flatten()
    trend_vals = trend_vals[~np.isnan(trend_vals)]
    total      = len(trend_vals)    

    # Beräkna andel per klass
    classes = {
        "Ökande p<0.01 (%)":      4,
        "Ökande p<0.05 (%)":      3,
        "Ökande p<0.10 (%)":      2,
        "Ökande ej sign. (%)":    1,
        "Minskande ej sign. (%)": -1,
        "Minskande p<0.10 (%)":   -2,
        "Minskande p<0.05 (%)":   -3,
        "Minskande p<0.01 (%)":   -4,
    }
    class_pcts = {label: round(np.sum(trend_vals == val) / total * 100, 1)
                  for label, val in classes.items()}


    summary_results.append({
        "Område":                   area,
        "Index":                    index_name,
        "Median slope":             round(float(np.nanmedian(slope_vals)), 5),
        "Median Tau":               round(float(np.nanmedian(tau_vals)), 3),
        "Andel signifikanta (%)":   round(pct_significant, 1),
        "Andel ökande sig. (%)":    round(pct_increasing, 1),
        "Andel minskande sig. (%)": round(pct_decreasing, 1),
        "Dominant trend":           dominant_trend,
        **class_pcts,
    })

df_summary = pd.DataFrame(summary_results)

# Formatera och skriv ut
display(df_summary.style
    .format({
        "Median slope":             "{:.5f}",
        "Median Tau":               "{:.3f}",
        "Andel signifikanta (%)":   "{:.1f}",
        "Andel ökande sig. (%)":    "{:.1f}",
        "Andel minskande sig. (%)": "{:.1f}",
        "Ökande p<0.01 (%)":        "{:.1f}",
        "Ökande p<0.05 (%)":        "{:.1f}",
        "Ökande p<0.10 (%)":        "{:.1f}",
        "Ökande ej sign. (%)":      "{:.1f}",
        "Minskande ej sign. (%)":   "{:.1f}",
        "Minskande p<0.10 (%)":     "{:.1f}",
        "Minskande p<0.05 (%)":     "{:.1f}",
        "Minskande p<0.01 (%)":     "{:.1f}",
    })
    .set_caption(f"MK/Theil-Sen-sammanfattning – {area}")
    .set_table_styles([{"selector": "caption", "props": [("font-weight", "bold"), ("font-size", "13px")]}])
)

# Spara CSV för detta studieområde
out_path = os.path.join(output_dir_theil_sen_mk, f"{area}_ts_mk_summary.csv")
df_summary.to_csv(out_path, index=False, encoding="utf-8-sig")
print(f"Sparad: {out_path}")

In [ ]:
# =============================================
# Konkatenera sammanfattningar från flera studieområden
# =============================================
# Kör notebooken för varje studieområde (ändra `area` i konfigurationscellen) och
# samla CSV-filerna här efteråt.

areas = ["GRM", "GM", "LF"]  # Uppdatera med alla studieområden när de körts, t.ex. ["GS", "AM", "TAM"]
function = "obj" # "ref" eller "obj"
all_summaries = []
for a in areas:
    csv_path = os.path.join(output_dir_theil_sen_mk, f"{a}_ts_mk_summary.csv")
    if os.path.exists(csv_path):
        all_summaries.append(pd.read_csv(csv_path, encoding="utf-8-sig"))
    else:
        print(f"OBS: Ingen fil hittad för {a} – kör notebooken för det studieområdet först.")

if all_summaries:
    df_all = pd.concat(all_summaries, ignore_index=True)
    display(df_all.style
        .format({
            "Median slope":             "{:.5f}",
            "Median Tau":               "{:.3f}",
            "Andel signifikanta (%)":   "{:.1f}",
            "Andel ökande sig. (%)":    "{:.1f}",
            "Andel minskande sig. (%)": "{:.1f}",
            "Ökande p<0.01 (%)":        "{:.1f}",
            "Ökande p<0.05 (%)":        "{:.1f}",
            "Ökande p<0.10 (%)":        "{:.1f}",
            "Ökande ej sign. (%)":      "{:.1f}",
            "Minskande ej sign. (%)":   "{:.1f}",
            "Minskande p<0.10 (%)":     "{:.1f}",
            "Minskande p<0.05 (%)":     "{:.1f}",
            "Minskande p<0.01 (%)":     "{:.1f}",
        })
        .set_caption("MK/Theil-Sen-sammanfattning – alla studieområden")
        .set_table_styles([{"selector": "caption", "props": [("font-weight", "bold"), ("font-size", "13px")]}])
    )
    combined_path = os.path.join(output_dir_theil_sen_mk, f"{function}_omraden_ts_mk_summary.csv")
    df_all.to_csv(combined_path, index=False, encoding="utf-8-sig")
    print(f"Sparad: {combined_path}")